# BT4012: Candidate for the third Kaggle submission

**Student number: E1156285**

This companion notebook preserves the original journal. It checks and trains a new candidate: an equal average of two Histogram Gradient Boosting models using different numerical bin sizes.

The competition asks us to predict illicit probabilities for future transactions. Its metric is **ROC-AUC**. Unknown labels are not legitimate labels and are excluded from supervised training.

Only competition-provided data are used. No external transaction labels are obtained or used.


## 1. Load the competition data

We load the training transactions, future test transactions and submission template. Each transaction has 165 anonymous numeric features.

Run this notebook from the Kaggle project folder containing `Data Set`. The original data files are read without modification.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from threadpoolctl import threadpool_limits

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "Data Set"
assert DATA_DIR.is_dir(), "Open this notebook from the Kaggle project folder."

train = pd.read_csv(DATA_DIR / "Train Data.csv")
test = pd.read_csv(DATA_DIR / "Test Data.csv")
sample_submission = pd.read_csv(
    DATA_DIR / "Sample Submission" / "BT4012 Competition Submission.csv"
)
FEATURE_COLUMNS = [f"feat_{i}" for i in range(1, 166)]

labels = pd.to_numeric(train["label"], errors="coerce")
assert labels.dropna().isin([0, 1]).all()
labeled = train.loc[labels.notna()].copy()
labeled["label"] = labels.loc[labels.notna()].astype(int)

assert train["txId"].is_unique and test["txId"].is_unique
assert set(train["txId"]).isdisjoint(set(test["txId"]))
assert train["time_step"].max() < test["time_step"].min()
assert np.isfinite(train[FEATURE_COLUMNS].to_numpy()).all()
assert np.isfinite(test[FEATURE_COLUMNS].to_numpy()).all()

print("Transactions with known labels:", len(labeled))
print("Test transactions:", len(test))
print("Features:", len(FEATURE_COLUMNS))


Transactions with known labels: 31235
Test transactions: 15329
Features: 165


## 2. Define the candidate

### What changes?

Our original Histogram Gradient Boosting model groups each numerical feature into up to **255 bins**. A bin groups nearby values together so the trees have fewer possible cut-off points to examine.

We keep the previously evaluated settings and train two versions:

- One with up to **63 bins** per feature.
- One with up to **127 bins** per feature.

Fewer bins provide coarser cut-off points. This may reduce dependence on fine numerical details that do not transfer well to future periods. It can also remove useful detail, which is why we test the change rather than assume it helps.

We average the two predicted probabilities with equal weight. For example, predictions of 0.8 and 0.6 become 0.7. This is an **ensemble**, meaning a combined prediction from more than one model.

We use the same 165 features, no transaction IDs as predictive features, and no classification threshold. Graph features and additional packages are not needed for this candidate.

### Settings we retain

- Up to 400 training rounds, each with learning rate 0.05.
- Up to 15 leaves per tree and at least 30 examples in each leaf.
- L2 regularisation of 10 to discourage extreme leaf updates.
- Early stopping using a 10% internal holdout drawn only from the training portion.
- Random seed 42 for repeatability.


In [2]:
def make_boosting_model(max_bins):
    return HistGradientBoostingClassifier(
        max_iter=400,
        learning_rate=0.05,
        max_leaf_nodes=15,
        min_samples_leaf=30,
        l2_regularization=10,
        max_bins=max_bins,
        early_stopping=True,
        validation_fraction=0.1,
        random_state=42,
    )

ENSEMBLE_BINS = (63, 127)


## 3. Check the candidate on later periods

We train fresh models for each practice exam. Earlier transactions provide the learning material; later transactions provide the answers used to check predictions.

| Check | Training steps | Validation steps |
|---|---|---|
| Earlier, fourteen steps ahead | 1–14 | 15–28 |
| Later, fourteen steps ahead | 1–21 | 22–35 |
| Original recent check | 1–28 | 29–35 |

The first two checks cover fourteen future time steps, like the actual test period 36–49. They have smaller training sets than the final model, so they do not precisely predict the final score.

We compare the new ensemble directly with the original 255-bin configuration on the same examples. Each model learns only from known labels in its training portion. Its internal early-stopping check also stays within that portion.

These periods overlap and have already been used for model development. Their scores support a candidate choice; they are not untouched estimates of final performance.


In [3]:
validation_rows = []

for train_end, valid_start, valid_end in [(14, 15, 28), (21, 22, 35), (28, 29, 35)]:
    learning = labeled.loc[labeled["time_step"] <= train_end]
    checking = labeled.loc[labeled["time_step"].between(valid_start, valid_end)]
    assert learning["time_step"].max() < checking["time_step"].min()
    assert learning["label"].nunique() == checking["label"].nunique() == 2

    probabilities = {}
    for bins in (255, *ENSEMBLE_BINS):
        print(f"Training through step {train_end}, bins={bins}...", flush=True)
        model = make_boosting_model(bins)
        with threadpool_limits(limits=4):
            model.fit(learning[FEATURE_COLUMNS], learning["label"])
            illicit_column = list(model.classes_).index(1)
            probabilities[bins] = model.predict_proba(
                checking[FEATURE_COLUMNS]
            )[:, illicit_column]

    ensemble_probabilities = np.mean(
        [probabilities[bins] for bins in ENSEMBLE_BINS], axis=0
    )
    validation_rows.append({
        "training_steps": f"1–{train_end}",
        "validation_steps": f"{valid_start}–{valid_end}",
        "original_auc": roc_auc_score(checking["label"], probabilities[255]),
        "ensemble_auc": roc_auc_score(checking["label"], ensemble_probabilities),
    })

validation_comparison = pd.DataFrame(validation_rows)
validation_comparison["auc_change"] = (
    validation_comparison["ensemble_auc"] - validation_comparison["original_auc"]
)
display(validation_comparison.round(4))
print("Mean across the two fourteen-step checks:")
display(validation_comparison.iloc[:2][["original_auc", "ensemble_auc"]].mean().round(4))


Training through step 14, bins=255...


Training through step 14, bins=63...


Training through step 14, bins=127...


Training through step 21, bins=255...


Training through step 21, bins=63...


...

Training through step 28, bins=63...


Training through step 28, bins=127...


,training_steps,validation_steps,original_auc,ensemble_auc,auc_change
0,1–14,15–28,0.9448,0.9571,0.0123
1,1–21,22–35,0.9117,0.9150,0.0034
2,1–28,29–35,0.9950,0.9951,0.0001


Mean across the two fourteen-step checks:


original_auc    0.9282
ensemble_auc    0.9360
dtype: float64

### Interpretation of the local checks

In the reviewed run, the original model scored **0.9448, 0.9117 and 0.9950**. The two-model ensemble scored **0.9571, 0.9150 and 0.9951**.

The ensemble improved all three checks. Its equal-weight mean over the two fourteen-step checks was **0.9360**, versus **0.9282** for the original model.

The later fourteen-step check remains below 0.96. These results therefore support a modest improvement, **not a promise of exceeding the competition baseline**.

Other tested changes—additional graph summaries, alternative boosting libraries, longer training and five-seed averaging—did not offer a consistently better result across the longer checks. They are omitted from this candidate to keep the final approach understandable and reproducible.


## 4. Train on all known labels and predict the test transactions

We now train two fresh models using all 31,235 known labels from steps 1–35. We keep the settings chosen during validation and average their predictions for the 15,329 test transactions from steps 36–49.

Unknown training labels are still excluded. Each model uses its internal training holdout for early stopping. The test labels are hidden and are never used for training or choosing the settings.

The averaged outputs are illicit probability estimates. We retain these scores instead of converting them into 0/1 decisions because Kaggle evaluates ROC-AUC.


In [4]:
final_models = {}
test_probability_columns = []

for bins in ENSEMBLE_BINS:
    print(f"Training final model with {bins} bins...", flush=True)
    model = make_boosting_model(bins)
    with threadpool_limits(limits=4):
        model.fit(labeled[FEATURE_COLUMNS], labeled["label"])
        illicit_column = list(model.classes_).index(1)
        probabilities = model.predict_proba(test[FEATURE_COLUMNS])[:, illicit_column]
    final_models[bins] = model
    test_probability_columns.append(probabilities)
    print("Training rounds completed:", model.n_iter_)

test_probabilities = np.mean(test_probability_columns, axis=0)


Training final model with 63 bins...


Training rounds completed: 400
Training final model with 127 bins...


Training rounds completed: 376


## 5. Check and save the submission

We match predictions to the template by identifier rather than assume rows are already in the right order. We check that every required identifier has one finite probability between 0 and 1.

The CSV contains only `index` and `target`. The original submissions are preserved under their existing filenames.

Upload only this new CSV for the remaining submission. Record its public score afterwards. The public score covers 30% of test transactions; the private score covers the other 70%.


In [5]:
assert test["index"].is_unique
assert sample_submission["index"].is_unique
assert len(test) == len(sample_submission)
assert set(test["index"]) == set(sample_submission["index"])

probabilities_by_index = pd.Series(test_probabilities, index=test["index"])
submission = sample_submission[["index"]].copy()
submission["target"] = submission["index"].map(probabilities_by_index)

assert submission.columns.tolist() == ["index", "target"]
assert len(submission) == len(test)
assert submission["index"].equals(sample_submission["index"])
assert np.isfinite(submission["target"].to_numpy()).all()
assert submission["target"].between(0, 1).all()

submission_dir = PROJECT_DIR / "submissions"
submission_dir.mkdir(parents=True, exist_ok=True)
submission_path = submission_dir / "submission_hist_bin_ensemble.csv"
submission.to_csv(submission_path, index=False)

validation_comparison.to_csv(
    submission_dir / "hist_bin_ensemble_local_validation.csv", index=False
)
display(submission.head())
print("Submission rows:", len(submission))
print("Saved to:", submission_path)


,index,target
0,0,0.000849
1,1,0.002700
2,2,0.002558
3,3,0.000532
4,4,0.000291


Submission rows: 15329
Saved to: /Users/mlvlo/BT4012/Kaggle/submissions/submission_hist_bin_ensemble.csv


## 6. Record the Kaggle result

| Submission | Public ROC-AUC |
|---|---:|
| Original Histogram Gradient Boosting | 0.93654 |
| Extra Trees | 0.91673 |
| Two-model bin ensemble | Pending submission |

Do not replace a better submission simply because a new candidate is newer. Record the result and retain the previous files. Threshold testing remains a separate pending analysis and does not change ROC-AUC for fixed probabilities.

For the final assignment, incorporate the selected approach and its actual results into `e1156285.ipynb` and the report. This companion notebook is a reproducible candidate, not a claim that the final model has already exceeded 0.96.

Technical reference: [Scikit-learn histogram gradient boosting documentation](https://scikit-learn.org/stable/modules/ensemble.html#histogram-based-gradient-boosting).
